# 07 · SEI 상(相) 매핑 — halo·ring·spot 픽셀 분류 → phase map + 두께

흐름: **입력 → 전처리 → 개요(median/MAX/center) → radial(예상 peak) → Halo → Ring → Spot → Phase map(확정/예상/약함) → 두께**.
각 픽셀에서 radial integration을 해 halo(비정질)·ring(다결정)·spot(단결정)을 보고, 물질과 상을 **확정/예상/약함**으로 판정한다.
로직은 패키지(`fds.classify_pixels` 등), 이 노트북은 **각 셀 파라미터를 노출**해 데이터마다 조절한다.
플롯 텍스트는 ASCII(한글은 마크다운/print만). 판정: **확정**=스팟 인덱싱(격자 자기일관) / **예상**=링 지문 일치 / **약함**=물질이나 상 불명.

## 1) 입력 — 로드 (경로만 바꾸면 어느 데이터든)

In [ ]:
import os, numpy as np, matplotlib.pyplot as plt
import fourdstem as fds

DM4_PATH   = "/home/jonghoonk918/Desktop/fdstem/Amorphous/Li-SEI/P-Cu/P-Gu1.dm4"   # ★데이터셋
USE_SYNTHETIC = not os.path.isfile(DM4_PATH)
DET_BIN    = 1                       # 검출기 비닝(메모리). q_max는 안 변함
Q_UNIT_HINT= "1/nm"                 # dm 단위(0.043888 1/nm)
CANDIDATES = ["LiF","Li2O","Li3N","Li2CO3","Li2S"]
N_JOBS     = -1                      # 병렬 코어(-1=전부; 32코어면 32)
PHASE_COL  = {"LiF":"#2ca02c","Li2O":"#1f77b4","Li3N":"#9467bd","Li2CO3":"#ff7f0e","Li2S":"#8c564b"}

def _synth(Sy=22,Sx=30,H=88,W=88,seed=0):
    rng=np.random.default_rng(seed); yy,xx=np.mgrid[0:H,0:W]; cx,cy=W/2,H/2; rr=np.hypot(xx-cx,yy-cy)
    beam=8*np.exp(-rr**2/8); halo=lambda r0,s=4:np.exp(-(rr-r0)**2/(2*s**2))
    def sp(r0,n=6,a=4):
        im=np.zeros((H,W))
        for k in range(n):
            t=2*np.pi*k/n; im+=a*np.exp(-((xx-cx-r0*np.cos(t))**2+(yy-cy-r0*np.sin(t))**2)/(2*1.6**2))
        return im
    cube=np.empty((Sy,Sx,H,W),np.float32)
    for iy in range(Sy):
        for ix in range(Sx):
            base=0.9*beam if iy>=Sy-3 else (beam+sp(20)+0.5*halo(20) if ix<Sx//2 else beam+1.2*halo(18))
            cube[iy,ix]=np.clip(base+0.15*rng.standard_normal((H,W)),0,None)
    return cube
if USE_SYNTHETIC: cube=fds.from_array(_synth(),q_per_px=0.02,name="synthetic")
else:
    cube=fds.load(DM4_PATH,Q_UNIT_HINT)
    if DET_BIN>1: cube=fds.bin_cube_detector(cube,DET_BIN)
scan=cube.scan_shape; QPP=cube.calibration.q_per_px
NAME=("synthetic" if USE_SYNTHETIC else os.path.splitext(os.path.basename(DM4_PATH))[0])
SAVE_DIR=("nb7_outputs" if USE_SYNTHETIC else os.path.dirname(DM4_PATH)+"/nb7_outputs"); os.makedirs(SAVE_DIR,exist_ok=True)
def save(fig,n): p=os.path.join(SAVE_DIR,f"{NAME}_{n}.png"); fig.savefig(p,dpi=150,bbox_inches="tight"); print("saved:",p)
def save_csv(n,h,rows):
    import csv; p=os.path.join(SAVE_DIR,f"{NAME}_{n}.csv")
    with open(p,"w",newline="") as f: w=csv.writer(f); w.writerow(h); w.writerows(rows)
    print("saved:",p)
print("cube:",cube.shape,"| q_per_px=",QPP,"| cores:",os.cpu_count(),"->",os.path.abspath(SAVE_DIR))

## 2) 전처리 — 진단(측정 먼저, 필요할 때만 교정)

인공물(중심/wander/타원/defect)은 교정, 신호 평활(블러)은 금지. 여기선 중심과 상태를 진단한다.

In [ ]:
# --- 이 셀 파라미터 ---
HOT_THRESHOLD  = 8.0    # hot/dead 검출 민감도(작을수록 민감). 진짜 스팟이 지워지면 키우기
WANDER_WARN_PX = 1.0    # 이보다 크면 per-position 정렬 권고
ELLIP_WARN     = 0.02   # 타원율 이보다 크면 타원 보정 권고
diag=fds.diagnose_cube(cube,hot_threshold=HOT_THRESHOLD); center=diag["center"]
print("=== 전처리 진단 ===")
print(f"  center           = ({center[0]:.1f},{center[1]:.1f})  (hot-pixel 제거 후 무게중심)")
print(f"  beam wander      = {diag['wander_px']:.2f} px  (>{WANDER_WARN_PX} 이면 정렬 고려)")
print(f"  detector defects = {100*diag['bad_pixel_frac']:.2f} %")
print(f"  ring ellipticity = {100*diag['ellipticity']:.1f} % @ {diag['ellipse_angle_deg']:.0f}deg  (>{100*ELLIP_WARN:.0f}% 면 보정)")
for n in diag["notes"]: print("  -",n)

## 3) 개요 — median / MAX NBD / center

median=비정질 halo가 잘 보임, MAX=다결정 링·스팟이 모여 보임.

In [ ]:
# --- 이 셀 파라미터 ---
CENTER = None       # None=진단값. 수동이면 (cx,cy)
if CENTER is not None: center=CENTER
med=fds.median_pattern(cube); mx=fds.clean_pattern(np.asarray(cube.max_dp(),float),hot_threshold=HOT_THRESHOLD)
fig,ax=plt.subplots(1,2,figsize=(9,4.4))
ax[0].imshow(np.log1p(med),cmap="magma"); ax[0].plot(*center,"c+",ms=9); ax[0].set_title("median NBD (log) - amorphous halo"); ax[0].axis("off")
ax[1].imshow((np.clip(mx,0,None)/mx.max())**0.3,cmap="magma"); ax[1].plot(*center,"c+",ms=9); ax[1].set_title("MAX NBD (gamma) - rings/spots"); ax[1].axis("off")
plt.tight_layout(); save(fig,"03_overview"); plt.show()

## 4) Radial integration — 예상되는 물질의 peak 위치 표시

전체 평균/최대 NBD의 방위각 적분에 **후보 5상의 링 위치**를 겹쳐 표시하고, 비정질 **FSDP**(halo 봉우리)를 찾는다.

In [ ]:
# --- 이 셀 파라미터 ---
RING_QBEAM=0.20; RING_QMAX=1.0; RING_NSIG=1.5; RING_TOPN=8   # 링 검출
qd,Id=fds.azimuthal_integrate(med,center,q_per_px=QPP)          # median -> halo
qm,Im=fds.azimuthal_integrate(mx,center,q_per_px=QPP)           # max -> rings
halo_q,halo_conf=fds.find_fsdp(qd,Id,q_lo=max(RING_QBEAM,0.10),q_hi=RING_QMAX)
rings_q=fds.detect_rings(mx,center,QPP,q_beam=RING_QBEAM,q_max=RING_QMAX,nsig=RING_NSIG,top_n=RING_TOPN)
rings_d=[round(1/q,2) for q in rings_q]
print(f"amorphous FSDP: q={halo_q:.3f} d={1/halo_q:.2f}A conf {halo_conf:.1f}")
print(f"검출 링 d(A) = {rings_d}")
fig,ax=plt.subplots(1,1,figsize=(10,4.2))
ax.semilogy(qm,np.clip(Im,1e-2,None),"k-",lw=0.9,label="MAX I(q)")
ax.semilogy(qd,np.clip(Id,1e-2,None),"0.5",lw=0.8,label="median I(q)")
for c in CANDIDATES:
    for dd,w in fds.COMPOUND_RINGS[c]:
        ax.axvline(1/dd,color=PHASE_COL[c],ls="--",lw=0.4+0.9*w,alpha=0.5)
for q in rings_q: ax.axvline(q,color="k",ls=":",lw=0.7)
ax.axvline(halo_q,color="r",ls="-",lw=1.2,label=f"FSDP d={1/halo_q:.2f}")
import matplotlib.patches as mp
ax.legend(handles=[mp.Patch(color=PHASE_COL[c],label=c) for c in CANDIDATES]+
          [plt.Line2D([],[],color="k",ls=":",label="detected ring"),plt.Line2D([],[],color="r",label="FSDP")],fontsize=7,ncol=2)
ax.set_xlabel("q (1/A)"); ax.set_ylabel("I(q)"); ax.set_title("radial integration + expected phase rings (dashed=candidate d)")
plt.tight_layout(); save(fig,"04_radial_expected"); plt.show()

## 5) Halo 분석 — 물질/진공 판정 (진공 대비 cutoff)

**환형 detector + 진공 cutoff.** 비정질 halo(broad diffuse) 반경들(FSDP + secondary)에 환형 detector를 놓고,
그 반경의 **peak-above-flank**(국소 배경 위 봉우리)를 **진공(무물질) 대비 몇 σ**인지 본다 — 진공이 노이즈 바닥이니
**N σ 이상이면 진짜 산란(물질)**, 아니면 가짜. 이게 **물질 마스크**(뒤 단계 재사용). halo는 비정질이라 상 이름은 안 붙임.
아래: 대표 halo 픽셀 NBD, **검출된 halo 반경별 significance 맵**(secondary가 살짝 보이는지), 물질 마스크.

In [ ]:
# --- 이 셀 파라미터 ---
VAC_PCTL      = 15          # 엄격 기준진공 = 총세기 하위 X% (detector cutoff 기준). 물질이 화면을 많이 채우면 낮추기
HALO_BAND     = (0.15,0.60) # 물질 판정용 broad halo 밴드(1/A) — 이 구간 산란이 진공 대비 유의하면 물질
HALO_SIGMA    = 3.0         # 진공 대비 이 σ 이상 = 진짜 물질(cutoff). ★물질이 과하게 잡히면 키우기(4,5)
HALO_STRONG_S = 8.0         # 이 σ 이상 = 확정물질(강 halo), 그 사이는 예상물질
HALO_DQ       = 0.04        # 개별 halo 반경 detector 반폭(그림용, 1/A)
NBIN          = 200
# 픽셀 분석(§6,7,8) 공통 파라미터
RING_THR=0.25; RING_MARGIN=1.15; CLS_QLO=0.30; CLS_SPOT_PCTL=95; CLS_SPOT_NMAD=8.0
# 1) 픽셀 radial stack 1회 + 엄격 진공
q_stack,prof=fds.radial_stack(cube,center,QPP,q_max=1.2,nbin=NBIN,n_jobs=N_JOBS)
vac_ref=fds.strict_vacuum_mask(cube,center=center,q_per_px=QPP,pctl=VAC_PCTL)
# 2) 물질 판정 = broad halo 밴드 전체를 진공 대비(plain 환형; 정확한 peak 불필요, robust)
q0b=0.5*(HALO_BAND[0]+HALO_BAND[1]); dqb=0.5*(HALO_BAND[1]-HALO_BAND[0])
halo_sig,_=fds.detector_map(cube,center,QPP,q0b,dq=dqb,flank=None,vacuum_mask=vac_ref,_stack=(q_stack,prof))
material=halo_sig>HALO_SIGMA
halo_tier=np.where(~material,0,np.where(halo_sig>=HALO_STRONG_S,3,2))
print(f"물질 {100*material.mean():.0f}% (halo밴드 {HALO_BAND}>{HALO_SIGMA}σ) | 확정물질(>{HALO_STRONG_S}σ) {int((halo_tier==3).sum())}px | 예상물질 {int((halo_tier==2).sum())}px | 진공 {int((~material).sum())}px")
# 개별 halo 반경(그림용): §4 FSDP + 고정 secondary 프로브
halo_probes=[p for p in ([halo_q,0.30,0.40,0.80]) if 0.1<p<1.0]
sig_maps=[fds.detector_map(cube,center,QPP,p,dq=HALO_DQ,flank=None,vacuum_mask=vac_ref,_stack=(q_stack,prof))[0] for p in halo_probes]
print("개별 halo 프로브 q =",[round(p,3) for p in halo_probes]," (진공 대비 유의픽셀 %:",[round(100*(sg>HALO_SIGMA).mean(),1) for sg in sig_maps],")")
# 3) 하류(ring/spot/phase)용 픽셀 분석 — 이 detector 물질을 사용
pc=fds.classify_pixels(cube,center=center,q_per_px=QPP,candidates=CANDIDATES,material=material,
      halo_q=halo_q,q_lo=CLS_QLO,ring_thr=RING_THR,margin=RING_MARGIN,
      spot_pctl=CLS_SPOT_PCTL,spot_kwargs=dict(n_mad=CLS_SPOT_NMAD,min_dist=3,tophat=11,q_max=1.15),
      index_kwargs=dict(tol_g=0.03,tol_ang=6.0,min_score=0.6,min_complete=0.6,confirm_min_spots=4),n_jobs=N_JOBS)
# --- 그림 A: 대표 halo NBD | halo band significance | 물질 마스크 ---
iy,ix=np.unravel_index(int(np.argmax(np.where(material,halo_sig,-np.inf))),scan)
fig,ax=plt.subplots(1,3,figsize=(14,4.3))
sel=np.zeros(scan,bool); sel[iy,ix]=True; pat=fds.average_pattern(cube,sel)
ax[0].imshow((np.clip(pat,0,None)/np.max(pat))**0.3,cmap="magma"); ax[0].plot(*center,"c+",ms=8)
ax[0].add_patch(plt.Circle(center,HALO_BAND[0]/QPP,fill=False,ec="cyan",ls="--",lw=0.9,alpha=0.7))
ax[0].add_patch(plt.Circle(center,HALO_BAND[1]/QPP,fill=False,ec="cyan",ls="--",lw=0.9,alpha=0.7))
ax[0].set_title(f"representative HALO pixel ({iy},{ix})\nhalo band (cyan)"); ax[0].axis("off")
im=ax[1].imshow(np.where(material,halo_sig,np.nan),cmap="viridis"); plt.colorbar(im,ax=ax[1],fraction=0.046)
ax[1].set_title("halo-band significance (sigma vs vacuum)"); ax[1].axis("off")
from matplotlib.colors import ListedColormap,BoundaryNorm
cmH=ListedColormap(["#000000","#4575b4","#2ca02c"]); nmH=BoundaryNorm([-.5,.5,2.5,3.5],3)
ax[2].imshow(halo_tier,cmap=cmH,norm=nmH); ax[2].set_title(f"material (>{HALO_SIGMA}s): vacuum/weak/strong"); ax[2].axis("off")
plt.tight_layout(); save(fig,"05_halo_material"); plt.show()
# --- 그림 B: 개별 halo 반경별 significance 맵 (secondary가 보이는지) ---
nH=len(halo_probes)
fig,ax=plt.subplots(1,nH,figsize=(3.4*nH,3.6)); ax=np.atleast_1d(ax)
for a,p,sg in zip(ax,halo_probes,sig_maps):
    im=a.imshow(np.where(material,sg,np.nan),cmap="inferno"); plt.colorbar(im,ax=a,fraction=0.046)
    a.set_title(f"halo q={p:.2f} (d={1/p:.2f}A)\nsigma vs vacuum"); a.axis("off")
fig.suptitle("per-halo-radius significance (structure above vacuum)")
plt.tight_layout(); save(fig,"05_halo_peaks"); plt.show()
save_csv("05_halo",["halo_probe_q","halo_d","frac_above_3sigma"],
         [[f"{p:.4f}",f"{1/p:.3f}",f"{(sg>HALO_SIGMA).mean():.4f}"] for p,sg in zip(halo_probes,sig_maps)]
         +[["band_%.2f-%.2f"%HALO_BAND,"",f"{material.mean():.4f}"]])

## 6) Ring 분석 — 다결정 링으로 상 예상 (+ 확정/예상/약함)

각 픽셀 radial의 sharp 링을 **후보 상 지문(모든 링+강도비)**과 매칭한다. 지문이 한 상에 뚜렷하면 그 상을 **예상**으로.
대표 ring 픽셀 NBD(매칭 상의 예상 링 원 표시)와 **상별 ring 예상 맵**을 보여준다. (링만으로는 예상까지; 확정은 spot 인덱싱.)

In [ ]:
# --- 이 셀 파라미터 ---
RING_STRONG   = 0.45   # ring 점수 이 이상 = 강한 예상(그 이하 매칭은 약한 예상)
ring_pred=(pc.tier>=2)&material              # 링 지문이 상을 지정한 픽셀
print("=== ring 지문 상 예상 ===")
for k,c in enumerate(CANDIDATES):
    npx=int(((pc.phase_idx==k)&ring_pred).sum())
    if npx: print(f"  {c:7s}: {npx} px  (평균 score {pc.phase_score[(pc.phase_idx==k)&ring_pred].mean():.2f})")
iy,ix=pc.examples["ring"]; kbest=pc.phase_idx[iy,ix]
fig,ax=plt.subplots(1,3,figsize=(14,4.3))
sel=np.zeros(scan,bool); sel[iy,ix]=True; pat=fds.average_pattern(cube,sel)
ax[0].imshow((np.clip(pat,0,None)/np.max(pat))**0.3,cmap="magma"); ax[0].plot(*center,"c+",ms=8)
if kbest>=0:
    for dd,w in fds.COMPOUND_RINGS[CANDIDATES[kbest]]:
        ax[0].add_patch(plt.Circle(center,(1/dd)/QPP,fill=False,ec=PHASE_COL[CANDIDATES[kbest]],ls="--",lw=0.4+0.9*w,alpha=0.7))
ax[0].set_title(f"representative RING pixel ({iy},{ix})\nbest match: {CANDIDATES[kbest] if kbest>=0 else 'none'}"); ax[0].axis("off")
im=ax[1].imshow(np.where(material,pc.phase_score,np.nan),cmap="inferno"); plt.colorbar(im,ax=ax[1],fraction=0.046)
ax[1].set_title("ring fingerprint score (best phase)"); ax[1].axis("off")
from matplotlib.colors import ListedColormap,BoundaryNorm
cmP=ListedColormap([PHASE_COL[c] for c in CANDIDATES]); nmP=BoundaryNorm(np.arange(-.5,len(CANDIDATES)+.5),len(CANDIDATES))
lay=np.where(ring_pred&(pc.phase_idx>=0),pc.phase_idx,np.nan)
ax[2].imshow(np.where(material,0.12,np.nan),cmap="Greys",vmin=0,vmax=1)
ax[2].imshow(lay,cmap=cmP,norm=nmP)
import matplotlib.patches as mp
ax[2].legend(handles=[mp.Patch(color=PHASE_COL[c],label=c) for c in CANDIDATES],fontsize=7,loc="lower right",framealpha=.6)
ax[2].set_title("ring-predicted phase map"); ax[2].axis("off")
plt.tight_layout(); save(fig,"06_ring"); plt.show()

## 7) Spot 분석 — 단결정 스팟 인덱싱 (확정)

방위각 spottiness가 높은 픽셀(단결정 grain)에서 스팟을 검출해 **인덱싱**(|g|+각도가 한 격자·정대축에 자기일관)한다.
인덱싱되면 **확정**. 대표 spot 픽셀 NBD(검출 스팟 표시)와 spottiness 맵 + 확정 위치를 보여준다.

In [ ]:
# --- 이 셀 파라미터 ---
SPOT_SHOW_NMAD=CLS_SPOT_NMAD    # 대표 픽셀 스팟 표시 문턱
idxres=[r for r in pc.indexed if r["best"] is not None]
conf=[r for r in pc.indexed if r["best"] and r["best"].get("indexed")]
print(f"=== spot 인덱싱: seed {len(pc.indexed)}개 확인, 확정 {len(conf)}개 ===")
for r in sorted(pc.indexed,key=lambda d:-(d['best']['n_matched'] if d['best'] else 0))[:12]:
    iy,ix=r["pos"]; b=r["best"]
    if b is None: print(f"  ({iy:2d},{ix:2d}) spots{r['n_spots']:2d}: 스팟<2"); continue
    tag="[확정]" if b["indexed"] else ("[예상]" if b["n_matched"]>=2 else "[약함]")
    print(f"  ({iy:2d},{ix:2d}) spots{r['n_spots']:2d}: {tag} {b['phase']:6s} zone{str(b['zone']):12s} m{b['n_matched']}/{b['n_total']} comp{b['completeness']:.2f}")
print(f"  => 인덱싱 확정 상: {sorted({r['best']['phase'] for r in conf}) or '없음'}")
iy,ix=pc.examples["spot"]
fig,ax=plt.subplots(1,3,figsize=(14,4.3))
sel=np.zeros(scan,bool); sel[iy,ix]=True; pat=fds.average_pattern(cube,sel)
sp=fds.detect_spots(pat,center,QPP,n_mad=SPOT_SHOW_NMAD,min_dist=3,tophat=11,q_max=1.15)
ax[0].imshow((np.clip(pat,0,None)/np.max(pat))**0.3,cmap="gray"); ax[0].plot(*center,"c+",ms=8)
ax[0].scatter([s[0] for s in sp],[s[1] for s in sp],s=30,facecolors="none",edgecolors="yellow",lw=0.8)
ax[0].set_title(f"representative SPOT pixel ({iy},{ix})\n{len(sp)} spots"); ax[0].axis("off")
im=ax[1].imshow(np.where(material,pc.spot,np.nan),cmap="inferno"); plt.colorbar(im,ax=ax[1],fraction=0.046)
ax[1].set_title("spottiness (azimuthal variance)"); ax[1].axis("off")
ax[2].imshow(np.where(material,0.12,np.nan),cmap="Greys",vmin=0,vmax=1)
for r in pc.indexed:
    yy,xx=r["pos"]; b=r["best"]
    col="#2ca02c" if (b and b["indexed"]) else ("#ff7f0e" if (b and b["n_matched"]>=2) else "#d62728")
    ax[2].plot(xx,yy,"o",mfc="none",mec=col,ms=7,mew=1.3)
    if b and b["indexed"]: ax[2].text(xx+.5,yy,b["phase"],color=col,fontsize=6)
ax[2].set_xlim(-1,scan[1]); ax[2].set_ylim(scan[0],-1)
ax[2].set_title("spot seeds: green=indexed, orange=predict, red=weak"); ax[2].axis("off")
plt.tight_layout(); save(fig,"07_spot"); plt.show()

## 8) Phase mapping — 확정 / 예상 / 약함 (종합)

halo/ring/spot을 종합한 픽셀별 판정: **확정**(스팟 인덱싱) · **예상**(링 지문) · **약함**(물질이나 상 불명) · **없음**(진공).
상별 색으로 확정/예상/약함 맵을 보여준다.

In [ ]:
# --- 이 셀 파라미터 --- (판정은 §5~7에서 계산된 pc.tier 사용)
print("=== 픽셀 종합 판정 ===")
for t,name in [(3,"확정"),(2,"예상"),(1,"약함"),(0,"없음")]: print(f"  {name}: {int((pc.tier==t).sum())} px",end="")
print()
for k,c in enumerate(CANDIDATES):
    cf=int(((pc.phase_idx==k)&(pc.tier==3)).sum()); pr=int(((pc.phase_idx==k)&(pc.tier==2)).sum())
    if cf or pr: print(f"    {c:7s}: 확정 {cf} px, 예상 {pr} px")
conf_ph=sorted({CANDIDATES[k] for k in range(len(CANDIDATES)) if ((pc.phase_idx==k)&(pc.tier==3)).any()})
pred_ph=sorted({CANDIDATES[k] for k in range(len(CANDIDATES)) if ((pc.phase_idx==k)&(pc.tier==2)).any()})
print(f"  => 확정 상: {conf_ph or '없음'} | 예상 상: {pred_ph or '없음'}")
from matplotlib.colors import ListedColormap,BoundaryNorm
import matplotlib.patches as mp
cmP=ListedColormap([PHASE_COL[c] for c in CANDIDATES]); nmP=BoundaryNorm(np.arange(-.5,len(CANDIDATES)+.5),len(CANDIDATES))
def layer(t): return np.where((pc.tier==t)&(pc.phase_idx>=0),pc.phase_idx,np.nan)
fig,ax=plt.subplots(1,3,figsize=(15,4.4))
for a,(t,ttl) in zip(ax,[(3,"CONFIRMED (indexed)"),(2,"PREDICTED (ring)"),(1,"WEAK (material, no phase)")]):
    a.imshow(np.where(material,0.12,np.nan),cmap="Greys",vmin=0,vmax=1)
    if t==1: a.imshow(np.where(pc.tier==1,1.0,np.nan),cmap="Greys",vmin=0,vmax=1.4)
    else: a.imshow(layer(t),cmap=cmP,norm=nmP)
    a.set_title(f"{ttl}: {int((pc.tier==t).sum())} px"); a.axis("off")
ax[0].legend(handles=[mp.Patch(color=PHASE_COL[c],label=c) for c in CANDIDATES],fontsize=7,loc="lower right",framealpha=.6)
fig.suptitle("PHASE MAP - CONFIRMED / PREDICTED / WEAK (color = phase)")
plt.tight_layout(); save(fig,"08_phase_map"); plt.show()
save_csv("08_phase_tiers",["phase","confirmed_px","predicted_px"],
         [[c,int(((pc.phase_idx==CANDIDATES.index(c))&(pc.tier==3)).sum()),int(((pc.phase_idx==CANDIDATES.index(c))&(pc.tier==2)).sum())] for c in CANDIDATES])

## 9) 두께 계산 — t/lambda = ln(I_total / I_beam)

앞에서 **물질/진공을 이미 판정**했으니 이를 활용한다. 진공(=물질 없는 곳)을 기준으로 dark를 자기교정하고 영점을 맞춘 **상대 두께**.
낮음=얇음(표면/껍데기), 높음=두꺼움. (수렴빔·각도분리라 절대 nm은 λ 필요 → 상대값만 신뢰.)

In [ ]:
# --- 이 셀 파라미터 ---
BEAM_RADIUS_PX=None    # 직접빔 디스크 반경(px). None=자동(~det/20)
VAC_PCTL=15            # 총세기 하위 X% = 엄격 기준진공(dark/영점용, 확실히 빈 곳). 물질이 많으면 낮추기
_flat=cube._flat_patterns(); _tot=np.asarray(_flat,float).reshape(_flat.shape[0],-1).sum(1)
vac_ref=np.asarray(_tot<=np.percentile(_tot,VAC_PCTL),bool).reshape(scan)   # 엄격 기준진공(오염 방지)
tmap,tex=fds.thickness_map(cube,center=center,beam_radius=BEAM_RADIUS_PX,vacuum_mask=vac_ref,return_extras=True)
tin=tmap[material]; tin=tin[np.isfinite(tin)]; tvac=tmap[vac_ref]; tvac=tvac[np.isfinite(tvac)]
print(f"dark 자기교정 D={tex['dark']:.3g} | 영점 offset={tex['offset']:.3g} | beam_r={tex['beam_radius']:.1f}px")
print(f"진공 t/lambda mean {tvac.mean():+.3f} (≈0 정상) | 물질 t/lambda mean {np.mean(tin):.2f} (5~95%: {np.percentile(tin,5):.2f}~{np.percentile(tin,95):.2f})")
fig,ax=plt.subplots(1,2,figsize=(11,4.2))
im=ax[0].imshow(np.where(material,tmap,np.nan),cmap="viridis"); plt.colorbar(im,ax=ax[0],fraction=0.046)
ax[0].set_title("relative thickness t/lambda (vacuum-referenced)"); ax[0].axis("off")
ax[1].hist(tin,bins=50,color="0.4",label="material"); ax[1].axvline(0,color="r",ls="--",lw=1,label="vacuum (0)")
ax[1].set_xlabel("t/lambda"); ax[1].set_ylabel("# positions"); ax[1].set_title("thickness distribution (thin <-> thick)"); ax[1].legend(fontsize=8)
plt.tight_layout(); save(fig,"09_thickness"); plt.show()
save_csv("09_thickness_stats",["metric","value"],
         [["dark",f"{tex['dark']:.4g}"],["vac_mean",f"{tvac.mean():.4f}"],["mat_mean",f"{np.mean(tin):.4f}"],
          ["mat_p05",f"{np.percentile(tin,5):.4f}"],["mat_p95",f"{np.percentile(tin,95):.4f}"],["material_frac",f"{material.mean():.4f}"]])
print("[정리] 확정=스팟 인덱싱된 상 / 예상=링 지문 / 약함=비정질·불명. 두께는 진공 대비 상대값.")